In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import sys, os
sys.path.append('../')
sys.path.append('../Benchmark/')
import MeshFEM
import mesh, mesh_energy
import energy
import parametrization, viewer, benchmark, py_newton_optimizer
import flip_avoiding_step_length

In [3]:
import param_utils
import helper_funcs

# Load Mesh

In [4]:
mesh_path = '../../../Models/TableOneModels/Hilbert2.off'

In [5]:
# m = mesh.Mesh('../../../Models/TableOneModels/cow2Disc.off')
# m = mesh.Mesh('../../../Models/TableOneModels/Lucy_3cuts.off')
m = helper_funcs.read_mesh(mesh_path)

In [6]:
print(f'mesh vertices: {m.numVertices()}; mesh elements: {m.numElements()}')

mesh vertices: 18424; mesh elements: 32228


# Initialization

In [7]:
uv = mesh_energy.NodalVars(m, 2)

In [8]:
# mesh initialization
# uv_init = parametrization.lscm(m)

bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
uv_init = parametrization.harmonic(m, bdry_uv)
flip_list = parametrization.getFlips(m, uv_init)
if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)


uv.setVars(uv_init.ravel())

In [9]:
psi = energy.SymmetricDirichlet(2)

param = mesh_energy.ParametrizationProjectToRestHessian(m, uv, psi)
param2 = mesh_energy.Parametrization(m, uv, psi)

H_current = param.hessian(projectionMask=False)
H_rest_projected = param.hessian(projectionMask=True)

In [10]:
H_param = param2.hessian(projectionMask=True)

In [11]:
H_rest_projected

In [12]:
import numpy as np

In [13]:
H_diff = np.linalg.norm(H_param.toSciPy().data - H_rest_projected.toSciPy().data)

In [14]:
H_diff / np.linalg.norm(H_rest_projected.toSciPy().data)

np.float64(217174847321462.84)

# UV viewer

In [15]:
uv_mesh = mesh.Mesh(uv.getVars().reshape(-1,2), m.elements())

In [16]:
uv_viewer = viewer.Viewer(uv_mesh, wireframe=True)
uv_viewer.show()

Renderer(camera=PerspectiveCamera(children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0), quaternion=(…

# Optimize

In [17]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [param])

# Work around energy nullspace by adding a small shift
prob.hessianShift = 1e-8 # 1e-8
opt = prob.optimizer()

In [18]:
# Applying flip-avoiding linesearch
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.8    # in accordance to Composite Majorization

In [19]:
opt.options.verboseNonPosDef = True
opt.options.niter = 200
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
# opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()

In [20]:
benchmark.reset()
cr = opt.optimize()
benchmark.report()

0	188092	8.44341e+10	8.11838e-13	0	1
1	3159.4	2.01997e+07	3.75856e-09	0	1
2	510.431	1.10357e+07	1.09809e-10	0	1
3	509.309	9.61584e+06	1.69037e-10	0	1
4	499.948	4.39032e+06	6.57878e-10	0	1
5	472.774	1.23256e+06	2.29116e-09	0	1
6	455.467	544514	5.99109e-09	0	1
7	425.685	234256	1.51489e-08	0	1
8	385.127	126470	3.96274e-08	0	1
9	336.416	73023	1.21701e-07	0	1
10	282.962	85868.3	4.8201e-07	0	1
11	227.351	14329.6	2.31838e-07	0	1
12	221.456	274644	6.4673e-09	0	1
13	219.598	12806.5	1.8704e-07	0	1
14	215.54	148628	8.51732e-09	0	1
15	214.213	8647.49	8.82727e-07	0	1
16	198.106	106550	5.38391e-08	0	1
17	197.127	31850.6	2.1221e-07	0	1
18	194.497	44841.9	1.17336e-07	0	1
19	193.105	23397.1	3.48301e-07	0	1
20	189.994	24608.4	1.0251e-07	0	1
21	189.222	62726.4	9.72411e-08	0	1
22	188.404	69186.1	1.23218e-07	0	1
23	186.906	8399.61	3.41529e-06	0	1
24	166.029	6802.14	2.77409e-06	0	1
25	158.726	10822.6	3.56777e-07	0	1
26	158.488	55279.9	2.49654e-08	0	1
27	157.862	4398.43	1.41545e-06	0	1
28	155.66	20073.7	9.22